# Real DES data: look, cut, save

Turns a DES catalogue you downloaded into the small table the rest of this
project reads. Every cell is meant to be run and looked at, not trusted.

**The output contract.** `build_raw_background_maps` and the matched filters
need exactly this, and nothing else:

| column | meaning |
|---|---|
| `ra`, `dec` | degrees |
| `des_yr6_g_obs`, `des_yr6_r_obs` | magnitudes, extinction-corrected, already cut and clipped |

The namespace `des_yr6` comes from `SURVEY`/`RELEASE` in the stream-parameters
experiment (`streamobs.columns.obs_col`). If you change survey or release,
this name changes with it.

**Where the data comes from** — check the schema yourself, these names drift:

- *Public, easiest*: NOIRLab Astro Data Lab, <https://datalab.noirlab.edu>.
  DES DR2 is the public six-year release. Browse the tables at
  `datalab.noirlab.edu/query.php` and write an ADQL/SQL query with a box in
  RA/Dec, selecting **only the columns below** — the full table is ~690
  million rows and you do not want it.
- *Public, bulk*: DES DR2 tiles as FITS from the NCSA release server, if you
  prefer files over a query service.
- *Collaboration*: Y6 Gold, if you have DES internal access. Better
  star/galaxy separation and extinction than DR2, and it is the release the
  `des`/`yr6` streamobs model is built for.

Columns to ask for from `des_dr2.main`: `ra`, `dec`, `mag_auto_g`,
`mag_auto_r`, `magerr_auto_g`, `magerr_auto_r`, `ext_coadd`, `flags_g`,
`flags_r`, and whatever extinction the schema offers.

`ext_coadd` is the coadd star/galaxy classifier: 0 high-confidence star,
1 candidate star, 2 mostly galaxy, 3 high-confidence galaxy, **-9 no data**.
`<= 1` therefore has to be `between(0, 1)`, or the -9 rows come along.

**Start small.** Pull a 5x5 degree box first, run this notebook end to end,
then go back for the real region. A 25x18 degree box is a few tens of
millions of rows before cuts.

## 1. Load what you downloaded

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path("~/Downloads/des_dr2_box.fits").expanduser()  # <- your file

if RAW.suffix == ".fits":
    from astropy.table import Table

    raw = Table.read(RAW).to_pandas()
else:
    raw = pd.read_parquet(RAW)

raw.columns = [c.lower() for c in raw.columns]
print(f"{len(raw):,} rows, {len(raw.columns)} columns")
raw.head()

In [ ]:
# What is actually in here: ranges, and how much is missing.
raw.describe().T[["count", "mean", "min", "max"]]

## 2. Where it is on the sky

`skyproj` draws the footprint properly. Look for holes, the survey edge, and
anything with obviously different density — those are the regions to avoid or
mask later.

In [ ]:
import matplotlib.pyplot as plt
import skyproj

fig, ax = plt.subplots(figsize=(9, 5))
sp = skyproj.McBrydeSkyproj(ax=ax)
sp.draw_hpxbin(raw["ra"].to_numpy(), raw["dec"].to_numpy(), nside=256, cmap="viridis")
sp.draw_colorbar(label="objects per pixel")
ax.set_title("everything downloaded, before any cut")
plt.show()

## 3. Colour-magnitude space

Two things to check: the stellar locus is where you expect it, and where the
counts fall off at the faint end — that is the real depth, which decides the
faint clip in the next cell.

In [ ]:
g, r = raw["mag_auto_g"], raw["mag_auto_r"]
finite = np.isfinite(g) & np.isfinite(r)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hexbin(
    (g - r)[finite], g[finite], gridsize=200, extent=(-0.5, 2.0, 16, 26),
    bins="log", cmap="magma",
)
axes[0].invert_yaxis()
axes[0].set_xlabel("g - r"); axes[0].set_ylabel("g"); axes[0].set_title("all objects")
axes[1].hist(g[finite], bins=120, range=(15, 27), histtype="step", label="g")
axes[1].hist(r[finite], bins=120, range=(15, 27), histtype="step", label="r")
axes[1].set_yscale("log"); axes[1].set_xlabel("magnitude"); axes[1].legend()
axes[1].set_title("where the counts turn over is the depth")
plt.show()

## 4. Cuts

Four kinds, in order of how much they remove. **Adjust the star/galaxy cut to
whatever your catalogue offers** — `ext_coadd` between 0 and 1 is the DR2
convention; Y6 Gold has its own classifier.

The magnitude clip must match the experiment's `CLIPPING` (16 to 24.5 in both
bands), because the matched filter and the trained models assume it. Apply it
**here, on dereddened magnitudes** — if you also cut server-side, cut wider
there (15.5 to 25), since extinction moves a star across a boundary by up to
a few tenths of a magnitude.

In [ ]:
CLIP = {"g": (16.0, 24.5), "r": (16.0, 24.5)}

# Extinction. DES uses A = coefficient x E(B-V); if your catalogue already has
# dereddened magnitudes, set both coefficients to 0 and point at those columns.
A_G, A_R = 3.186, 2.140
ebv = raw["ebv"] if "ebv" in raw else 0.0
g0 = raw["mag_auto_g"] - A_G * ebv
r0 = raw["mag_auto_r"] - A_R * ebv

keep = (
    np.isfinite(g0) & np.isfinite(r0)
    & (raw.get("ext_coadd", 0).between(0, 1))        # stars (0 or 1, not -9)
    & (raw.get("flags_g", 0) < 4) & (raw.get("flags_r", 0) < 4)
    & g0.between(*CLIP["g"]) & r0.between(*CLIP["r"])
)
print(f"kept {keep.sum():,} of {len(raw):,} ({100 * keep.mean():.1f}%)")

cut = pd.DataFrame({
    "ra": raw.loc[keep, "ra"].to_numpy(),
    "dec": raw.loc[keep, "dec"].to_numpy(),
    "des_yr6_g_obs": g0[keep].to_numpy(),
    "des_yr6_r_obs": r0[keep].to_numpy(),
})
cut.head()

## 5. The same two plots again

If the stellar locus did not tighten and the galaxies did not go, the
star/galaxy cut did not do what you think it did.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hexbin(
    cut["des_yr6_g_obs"] - cut["des_yr6_r_obs"], cut["des_yr6_g_obs"],
    gridsize=200, extent=(-0.5, 2.0, 16, 25), bins="log", cmap="magma",
)
axes[0].invert_yaxis(); axes[0].set_xlabel("g - r"); axes[0].set_ylabel("g")
axes[0].set_title("after cuts")
sp = skyproj.McBrydeSkyproj(ax=axes[1])
sp.draw_hpxbin(cut["ra"].to_numpy(), cut["dec"].to_numpy(), nside=256, cmap="viridis")
sp.draw_colorbar(label="stars per pixel")
axes[1].set_title("density after cuts: look for gradients")
plt.show()

## 6. Save

Parquet, so the next notebook opens it in seconds instead of re-reading the
download.

In [ ]:
OUT = Path("data/real/des_yr6_stars.parquet")
OUT.parent.mkdir(parents=True, exist_ok=True)
cut.to_parquet(OUT, index=False)
print(f"{len(cut):,} stars -> {OUT} ({OUT.stat().st_size / 1e6:.0f} MB)")

## 7. Does the pipeline accept it?

The real test of the contract: build one matched-filter map from this table
with the experiment's own filter. If this runs and the map has counts in it,
the file is usable as a background everywhere else.

In [ ]:
import importlib.util as u

spec = u.spec_from_file_location("sp_run", "scripts/experiments/stream_parameters/run.py")
run = u.module_from_spec(spec); spec.loader.exec_module(run)

from streamgoggles.background import build_raw_background_maps
from streamgoggles.matched_filter import PixelizationSpec, build_matched_filters

pix = PixelizationSpec(nside=512, image_size_pix=(96, 96))
filters = build_matched_filters(run.FILTERS, namespace=f"{run.SURVEY}_{run.RELEASE}")
maps = build_raw_background_maps(
    catalog=pd.read_parquet(OUT),
    matched_filters=filters,
    bands=["g", "r"],
    distance_moduli=[17.0],
    pix=pix,
)
raw_map, valid = maps["good"][17.0]
print(f"selected stars: {raw_map.sum():,.0f}")
print(f"covered pixels: {valid.sum():,} "
      f"(mean {raw_map[valid].mean():.2f} stars per pixel)")